# Epälineaaristen vähittäiskysyntäkäyrien mallintaminen PROC GAMilla

## Yhteenveto

Tässä muistikirjassa käytetään **PROC GAMia** mallintamaan viikoittaista
päivittäistavaramyyntiä sileänä, epälineaarisena funktiona hyllyhinnasta,
myymälän lämpötilasta (kausivaihtelun mittarina) ja kampanjakuluista, sekä
parametrisella kampanjalipun vaikutuksella. Yleistetyt additiiviset mallit
antavat tuoteryhmävastaavalle mahdollisuuden löytää todelliset kaartuvat
hintajouston ja kausikysynnän muodot, jotka lineaarinen regressio jättäisi
huomaamatta, mikä tukee tarkempaa hinnoittelua ja kampanjapäätöksiä.

## Tietolähteet

| Aineisto | Rivit | Taso | Avainmuuttujat | Kuvaus |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | viikko | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Synteettinen viikoittainen myyntisarja yhdelle lippulaivamyymälälle 100 peräkkäisen viikon ajalta (lähes kaksi kausisykliä). Luotu koodin sisällä komennoilla `call streaminit(20250531)` ja `rand()`. Viikoittainen yksikkömyynti noudattaa Poisson-kysyntäprosessia, jonka keskiarvoa ohjaavat eksponentiaalinen hintavastekäyrä, kvadraattinen lämpötila-/kausivaikutus huipentuen noin 72 asteen (°F) kohdalla, kovera (neliöjuuri-) kampanjakulujen nostovaikutus sekä diskreetti kampanjaviikon lippu. |

# Epälineaaristen vähittäiskysyntäkäyrien mallintaminen PROC GAMilla

Vähittäiskaupan kysyntä reagoi harvoin suoraviivaisesti hintaan, säähän tai
kampanjakuluihin. Pieni hinnanalennus perustuotteessa saattaa tuskin liikuttaa
myyntimäärää, kun taas psykologisen hintarajan ylittäminen voi laukaista
jyrkän hypyn; säävetoinen kysyntä huipentuu mukavassa keskialueessa ja laskee
molemmissa ääripäissä; ja kampanjan nostovaikutus osoittaa vähenevää tuottoa
kulujen kasvaessa.

**PROC GAM** (yleistetyt additiiviset mallit) sovittaa jokaiselle tekijälle
oman sileän splini-termin, joten data — ei jäykkä lineaarinen oletus —
määrittää jokaisen kysyntäkäyrän muodon. Tässä mallinnamme viikoittaista
yksikkömyyntiä yhdelle lippulaivamyymälälle 100 peräkkäisen viikon ajalta,
yhdistäen parametrisen kampanjalipun sileisiin splineihin hinnalle,
lämpötilalle ja kampanjakuluille Poisson-vasteen alla.

## Vaihe 1 — Luo synteettinen viikoittainen myyntisarja

Simuloimme 100 peräkkäistä viikkoa (lähes kaksi kausisykliä) myyntidataa
yhdelle lippulaivamyymälälle. Datan luontiprosessi on tarkoituksella
epälineaarinen, jotta voimme vahvistaa, että GAM löytää realistiset muodot:

- **Price** (hinta) ohjaa myyntimäärää eksponentiaalisen vastekäyrän kautta
  (`exp(-1.1 * Price)`), joten kysyntä nousee jyrkästi hinnan laskiessa.
- **Temperature** (lämpötila) toimii kausivaihtelun mittarina kvadraattisella
  huipulla noin 72 asteen (°F) kohdalla, mallintaen mukavuuteen perustuvaa
  asiakasvirtaa.
- **PromoSpend** (kampanjakulut) tuottaa koveran, neliöjuuripohjaisen
  nostovaikutuksen (vähenevä tuotto).
- Diskreetti **Promotion**-lippu lisää parametrisen askelvaikutuksen
  kampanjaviikkoina.

Viikoittainen `Units` (myyntimäärä) arvotaan Poisson-jakaumasta, mikä vastaa
yksikkömyynnin lukumääräluonnetta.

In [1]:
TIEDOT store_sales;
   CALL streaminit(20250531);
   PITUUS Promotion $3;
   TEE Week = 1 ASTI 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      JOS rand("uniform") < 0.28 NIIN TEE;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      LOPPU;
      MUUTEN TEE;
         Promotion  = "No";
         PromoSpend = 0;
      LOPPU;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      TULOSTE;
   LOPPU;
SUORITA;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Vaihe 2 — Profiloi simuloitu data

Nopea `PROC MEANS` vahvistaa ennen mallintamista, että muuttujat asettuvat
järkeville vähittäiskaupan arvoväleille: myyntimäärät ovat ei-negatiivisia
kokonaislukuja, hinta keskittyy muutaman dollarin tienoille, lämpötila kattaa
koko kausisyklin, ja kampanjakulut ovat nolla viikoilla, joina ei ole
kampanjaa.

In [2]:
PROSEDUURI KESKIARVOT TIEDOT=store_sales n mean MIN MAX maxdec=2;
   MUUTTUJA Units Price Temperature PromoSpend;
   NIMIKE Units="Myyntimäärä" Price="Hinta" Temperature="Lämpötila"
         PromoSpend="Kampanjakulut";
SUORITA;

                                                  The MEANS Procedure

 Variable     Label                  N           Mean     Minimum     Maximum
 ----------------------------------------------------------------------------
 Units        Myyntimäärä          100           7.03        0.00      103.00
 Price        Hinta                100           3.15        2.81        3.48
 Temperature  Lämpötila            100          55.50       22.72       83.49
 PromoSpend   Kampanjakulut        100         128.76        0.00      779.00
 ----------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 3 — Sovita täysi additiivinen kysyntämalli

Ydinmalli yhdistää:

- `param(Promotion)` — parametrinen (lineaarinen) vaikutus kampanjaviikon
  indikaattorille, ilmoitettu `CLASS`-lauseessa.
- `spline(Price, df=4)` — kuutiollinen sileä splini, joka vangitsee kaartuvan
  hintavasteen.
- `spline(Temperature, df=4)` — sileä kausivaihtelukäyrä.
- `spline(PromoSpend, df=3)` — vähenevän tuoton kampanjan nostovaikutus.

Koska myyntimäärät ovat lukumääriä, määritämme `dist=poisson` (log-linkki).
`method=gcv` antaa yleistetyn ristiinvalidoinnin ohjata minkä tahansa
komponentin sileyttä ilman eksplisiittistä vapausasteiden ohitusta.
`OUTPUT`-lause kirjoittaa havaintokohtaiset ennusteet ja jäännökset
`gam_fit`-aineistoon.

Proseduuri raportoi **devianssiksi 43,59** **nolladevianssia 2041,12**
vastaan — neljä sileää-plus-parametrista tekijää selittävät lähes kaiken
vaihtelun, jonka pelkkä vakiomalli jättäisi selittämättä — ja **AIC:ksi
254,61**. Parametrinen `PROMOTIONYES`-estimaatti on positiivinen (+0,41
logaritmisella asteikolla), mikä vahvistaa kampanjan nostovaikutuksen
myyntimäärään, ja hintaspline kantaa vahvasti negatiivisen lineaarisen
trendin (−1,71), joka on laskevan kysynnän tunnusmerkki.

In [3]:
PROSEDUURI gam TIEDOT=store_sales PLOTS=components(CLM commonaxes);
   LUOKKA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   NIMIKE Units="Myyntimäärä" Promotion="Kampanja" Price="Hinta"
         Temperature="Lämpötila" PromoSpend="Kampanjakulut";
   TULOSTE out=gam_fit predicted residual;
SUORITA;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Myyntimäärä
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Hinta)                    4.0000     


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Vaihe 4 — Suppeampi hinta- ja kausimalli

Kevyempää hinnoittelutarkastelua varten sovitamme mallin uudelleen käyttäen
vain kahta päätöksenteon kannalta tärkeintä sileää tekijää — hintaa ja
lämpötilaa — antaen hinnalle lisää joustavuutta (`df=5`) ratkaistaksemme
mahdollisen mutkan psykologisen hintarajan lähellä. Kampanjalippu säilytetään
parametrisena kontrollina.

Kampanjakulujen poistaminen nostaa **devianssin 62,80:aan** ja **AIC:n
269,82:een**, molemmat korkeampia kuin täyden mallin 43,59 ja 254,61.
Parametrinen `PROMOTIONYES`-termi absorboi tässä myös enemmän
kampanjasignaalia (+1,73 verrattuna +0,41:een), koska kampanjakulujen
splinettä ei enää ole kantamassa sitä. Hintaspline säilyttää negatiivisen
trendinsä (−1,51), joten kysynnän ydintarina pysyy vakaana eri
määrittelyissä.

In [4]:
PROSEDUURI gam TIEDOT=store_sales PLOTS=components(CLM);
   LUOKKA Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   NIMIKE Units="Myyntimäärä" Promotion="Kampanja" Price="Hinta"
         Temperature="Lämpötila";
SUORITA;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Myyntimäärä
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Hinta)                    5.0000         5.0000
Spline(Lämpötila)                4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Tulosten tulkinta

**Regression Model Analysis** -taulukko raportoi lineaarisen trendin jokaisen
komponentin sisällä sekä parametrisen kampanjavaikutuksen. Positiivinen
`PROMOTIONYES`-kerroin (+0,41 täydessä mallissa, +1,73 suppeammassa)
vahvistaa odotetun kampanjan myyntimäärän nostovaikutuksen, kun taas
hintasplinen negatiivinen lineaarinen trendi (−1,71 ja −1,51) heijastaa
klassista laskevaa kysyntää. Lämpötilasplinen pieni positiivinen
lineaaritermi (+0,03) on odotettu: sen todellinen tarina on kaarevuus 72
asteen (°F) mukavuushuipun ympärillä, jota yksi lineaarinen kerroin ei voi
tiivistää.

**Smoothing Model Analysis** -taulukko raportoi vapausasteet, jotka kukin
splini kuluttaa. Hinta ja lämpötila käyttävät kumpikin 4 efektiivistä
vapausastetta (5 hinnalle suppeammassa mallissa) ja kampanjakulut 3 — kaikki
selvästi enemmän kuin suoran viivan käyttämä yksi vapausaste, mikä on juuri
syy siihen, miksi lineaarinen regressio jättäisi nämä kaartuvat suhteet
huomaamatta.

**Fit Statistics** -taulukon avulla voidaan verrata kahta määrittelyä
suoraan. Täysi neljän tekijän malli raportoi devianssin 43,59 ja AIC:n
254,61 verrattuna suppeamman mallin 62,80 ja 269,82. Molemmat kriteerit
suosivat täyttä mallia, mikä osoittaa, että kampanjakulut ja kampanjalippu
tuovat selitysvoimaa pelkän hinnan ja lämpötilan lisäksi. Suhteessa
nolladevianssiin 2041,12 molemmat mallit vangitsevat valtaosan kysynnän
vaihtelusta.

Yhdessä luettuina nämä taulukot antavat tuoteryhmävastaavalle mitatun,
dataan perustuvan kysyntätarinan: jyrkkä hintavaste, joka ohjaa
alennussyvyyttä, kausiluonteinen lämpötilaikkuna ja vähenevän tuoton
kampanjakuluvaikutus — paljon tarkempi ohjenuora kuin yksi lineaarinen
jouston estimaatti. (PROC GAM hyväksyy myös `plots=components`-vaihtoehdon
osaennustekäyrien piirtämiseksi jokaiselle sileälle termille; yllä olevat
numeeriset komponenttitaulukot ovat lähde, josta nuo käyrät piirretään.)